# Cookidoo Agent Assistant - Initial Setup

This notebook sets up the foundational infrastructure for the Cookidoo Agent Assistant:
- Creates Unity Catalog, schemas, and tables
- Configures access permissions
- Initializes configuration tables
- Validates connections

**Prerequisites:**
- Databricks workspace with Unity Catalog enabled
- Catalog admin privileges
- Azure Key Vault configured with Cookidoo credentials

## 1. Install Required Libraries

In [ ]:
%pip install databricks-sdk mlflow azure-keyvault-secrets azure-identity

In [ ]:
dbutils.library.restartPython()

## 2. Configuration

In [ ]:
# Configuration variables
CATALOG_NAME = "cookidoo_agent"
SCHEMAS = {
    "main": "Main production schema for operational data",
    "staging": "Staging schema for data validation and testing",
    "analytics": "Analytics schema for aggregated insights",
    "ml": "Machine learning schema for models and features"
}

# Service principal or user groups
DATA_ENGINEERS_GROUP = "data_engineers"
DATA_SCIENTISTS_GROUP = "data_scientists"
ANALYSTS_GROUP = "analysts"

print(f"Catalog: {CATALOG_NAME}")
print(f"Schemas: {list(SCHEMAS.keys())}")

## 3. Create Catalog and Schemas

In [ ]:
%%sql

-- Create catalog if not exists
CREATE CATALOG IF NOT EXISTS cookidoo_agent
COMMENT 'Catalog for Cookidoo Agent Assistant application';

In [ ]:
%%sql

-- Use the catalog
USE CATALOG cookidoo_agent;

In [ ]:
%%sql

-- Create main schema
CREATE SCHEMA IF NOT EXISTS main
COMMENT 'Main production schema for operational data';

-- Create staging schema
CREATE SCHEMA IF NOT EXISTS staging
COMMENT 'Staging schema for data validation and testing';

-- Create analytics schema
CREATE SCHEMA IF NOT EXISTS analytics
COMMENT 'Analytics schema for aggregated insights';

-- Create ml schema
CREATE SCHEMA IF NOT EXISTS ml
COMMENT 'Machine learning schema for models and features';

## 4. Create Tables in Main Schema

In [ ]:
%%sql

USE SCHEMA main;

-- Recipes table
CREATE TABLE IF NOT EXISTS recipes (
  recipe_id STRING NOT NULL,
  title STRING NOT NULL,
  description STRING,
  servings INT,
  prep_time_minutes INT,
  cook_time_minutes INT,
  total_time_minutes INT,
  difficulty STRING,
  cuisine STRING,
  meal_type STRING,
  dietary_restrictions ARRAY<STRING>,
  ingredients ARRAY<STRUCT<
    name: STRING,
    quantity: DOUBLE,
    unit: STRING,
    notes: STRING
  >>,
  cooking_steps ARRAY<STRUCT<
    step_number: INT,
    instruction: STRING,
    duration_minutes: INT,
    temperature: STRING
  >>,
  nutritional_info STRUCT<
    calories: DOUBLE,
    protein_g: DOUBLE,
    carbs_g: DOUBLE,
    fat_g: DOUBLE,
    fiber_g: DOUBLE,
    sugar_g: DOUBLE,
    sodium_mg: DOUBLE
  >,
  source STRING,
  image_url STRING,
  tags ARRAY<STRING>,
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  PRIMARY KEY (recipe_id)
)
COMMENT 'Recipes catalog from Cookidoo and user modifications';

In [ ]:
%%sql

-- User interactions table
CREATE TABLE IF NOT EXISTS user_interactions (
  interaction_id STRING NOT NULL,
  user_id STRING,
  interaction_type STRING NOT NULL,
  recipe_id STRING,
  conversation_id STRING,
  request_text STRING,
  response_text STRING,
  modifications_applied ARRAY<STRING>,
  satisfaction_score DOUBLE,
  feedback STRING,
  timestamp TIMESTAMP NOT NULL,
  session_id STRING,
  metadata MAP<STRING, STRING>,
  PRIMARY KEY (interaction_id)
)
PARTITIONED BY (DATE(timestamp))
COMMENT 'User interactions with the LLM assistant';

In [ ]:
%%sql

-- Shopping lists table
CREATE TABLE IF NOT EXISTS shopping_lists (
  list_id STRING NOT NULL,
  user_id STRING,
  recipe_ids ARRAY<STRING>,
  items ARRAY<STRUCT<
    name: STRING,
    quantity: DOUBLE,
    unit: STRING,
    category: STRING,
    checked: BOOLEAN,
    notes: STRING,
    from_recipes: ARRAY<STRING>
  >>,
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  total_items INT,
  checked_items INT,
  PRIMARY KEY (list_id)
)
COMMENT 'Generated shopping lists from recipes';

In [ ]:
%%sql

-- Conversation history table
CREATE TABLE IF NOT EXISTS conversation_history (
  conversation_id STRING NOT NULL,
  user_id STRING,
  messages ARRAY<STRUCT<
    role: STRING,
    content: STRING,
    timestamp: TIMESTAMP
  >>,
  context MAP<STRING, STRING>,
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  message_count INT,
  PRIMARY KEY (conversation_id)
)
COMMENT 'Conversation history for chat sessions';

## 5. Create Analytics Tables

In [ ]:
%%sql

USE SCHEMA analytics;

-- Recipe popularity metrics
CREATE TABLE IF NOT EXISTS recipe_popularity (
  recipe_id STRING NOT NULL,
  date DATE NOT NULL,
  view_count INT DEFAULT 0,
  modification_count INT DEFAULT 0,
  save_count INT DEFAULT 0,
  average_rating DOUBLE,
  unique_users INT DEFAULT 0,
  PRIMARY KEY (recipe_id, date)
)
PARTITIONED BY (date)
COMMENT 'Daily recipe popularity metrics';

In [ ]:
%%sql

-- User engagement metrics
CREATE TABLE IF NOT EXISTS user_engagement (
  user_id STRING NOT NULL,
  date DATE NOT NULL,
  total_interactions INT DEFAULT 0,
  recipes_viewed INT DEFAULT 0,
  recipes_modified INT DEFAULT 0,
  recipes_saved INT DEFAULT 0,
  chat_messages INT DEFAULT 0,
  shopping_lists_created INT DEFAULT 0,
  session_duration_minutes DOUBLE,
  PRIMARY KEY (user_id, date)
)
PARTITIONED BY (date)
COMMENT 'Daily user engagement metrics';

## 6. Create ML Feature Tables

In [ ]:
%%sql

USE SCHEMA ml;

-- Recipe embeddings for vector search
CREATE TABLE IF NOT EXISTS recipe_embeddings (
  recipe_id STRING NOT NULL,
  title STRING,
  embedding ARRAY<DOUBLE>,
  embedding_model STRING,
  created_at TIMESTAMP,
  PRIMARY KEY (recipe_id)
)
COMMENT 'Recipe embeddings for similarity search';

In [ ]:
%%sql

-- User preference features
CREATE TABLE IF NOT EXISTS user_features (
  user_id STRING NOT NULL,
  preferred_cuisines ARRAY<STRING>,
  dietary_restrictions ARRAY<STRING>,
  skill_level STRING,
  avg_prep_time_preference INT,
  favorite_ingredients ARRAY<STRING>,
  disliked_ingredients ARRAY<STRING>,
  interaction_frequency_days DOUBLE,
  last_interaction TIMESTAMP,
  updated_at TIMESTAMP,
  PRIMARY KEY (user_id)
)
COMMENT 'User preference features for personalization';

## 7. Grant Permissions

In [ ]:
%%sql

-- Grant SELECT on all schemas to analysts
GRANT USE CATALOG ON CATALOG cookidoo_agent TO `analysts`;
GRANT USE SCHEMA ON SCHEMA cookidoo_agent.main TO `analysts`;
GRANT USE SCHEMA ON SCHEMA cookidoo_agent.analytics TO `analysts`;
GRANT SELECT ON SCHEMA cookidoo_agent.main TO `analysts`;
GRANT SELECT ON SCHEMA cookidoo_agent.analytics TO `analysts`;

-- Grant full access to data engineers
GRANT USE CATALOG ON CATALOG cookidoo_agent TO `data_engineers`;
GRANT ALL PRIVILEGES ON CATALOG cookidoo_agent TO `data_engineers`;

-- Grant ML schema access to data scientists
GRANT USE CATALOG ON CATALOG cookidoo_agent TO `data_scientists`;
GRANT USE SCHEMA ON SCHEMA cookidoo_agent.ml TO `data_scientists`;
GRANT ALL PRIVILEGES ON SCHEMA cookidoo_agent.ml TO `data_scientists`;

## 8. Validate Setup

In [ ]:
# Validate catalogs and schemas
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# List catalogs
print("=== Available Catalogs ===")
catalogs = spark.sql("SHOW CATALOGS").collect()
for catalog in catalogs:
    print(f"  - {catalog.catalog}")

# List schemas in cookidoo_agent
print("\n=== Schemas in cookidoo_agent ===")
schemas = spark.sql("SHOW SCHEMAS IN cookidoo_agent").collect()
for schema in schemas:
    print(f"  - {schema.databaseName}")

# List tables in main schema
print("\n=== Tables in cookidoo_agent.main ===")
tables = spark.sql("SHOW TABLES IN cookidoo_agent.main").collect()
for table in tables:
    print(f"  - {table.tableName}")

## 9. Create Configuration Table

In [ ]:
%%sql

USE SCHEMA main;

CREATE TABLE IF NOT EXISTS system_config (
  config_key STRING NOT NULL,
  config_value STRING,
  config_type STRING,
  description STRING,
  updated_at TIMESTAMP,
  updated_by STRING,
  PRIMARY KEY (config_key)
)
COMMENT 'System configuration key-value pairs';

In [ ]:
%%sql

-- Insert default configurations
MERGE INTO system_config AS target
USING (
  SELECT * FROM VALUES
    ('llm_model', 'claude-sonnet-4.5-20250514', 'string', 'Primary LLM model for recipe modifications'),
    ('llm_temperature', '0.7', 'float', 'Temperature for LLM responses'),
    ('max_tokens', '4096', 'int', 'Maximum tokens for LLM responses'),
    ('vector_search_index', 'recipe_embeddings_index', 'string', 'Name of the vector search index'),
    ('embedding_model', 'text-embedding-ada-002', 'string', 'Model for generating embeddings'),
    ('cache_ttl_seconds', '3600', 'int', 'Cache TTL in seconds'),
    ('rate_limit_per_minute', '60', 'int', 'API rate limit per minute'),
    ('version', '1.0.0', 'string', 'Application version')
  AS configs(config_key, config_value, config_type, description)
) AS source
ON target.config_key = source.config_key
WHEN NOT MATCHED THEN
  INSERT (config_key, config_value, config_type, description, updated_at, updated_by)
  VALUES (source.config_key, source.config_value, source.config_type, source.description, current_timestamp(), current_user());

In [ ]:
%%sql

SELECT * FROM system_config ORDER BY config_key;

## 10. Summary

In [ ]:
print("=" * 60)
print(" Setup Complete!")
print("=" * 60)
print(f"\nCatalog: {CATALOG_NAME}")
print(f"Schemas: {', '.join(SCHEMAS.keys())}")
print("\nTables created:")
print("  main:")
print("    - recipes")
print("    - user_interactions")
print("    - shopping_lists")
print("    - conversation_history")
print("    - system_config")
print("  analytics:")
print("    - recipe_popularity")
print("    - user_engagement")
print("  ml:")
print("    - recipe_embeddings")
print("    - user_features")
print("\nNext steps:")
print("  1. Run the DLT pipeline setup notebook")
print("  2. Configure vector search index")
print("  3. Deploy model endpoints")
print("  4. Set up AgentBricks agent")
print("=" * 60)